# 00 · Framing & dataset choice · **EXPLORE**

**Owner:** Jaime · **Status:** sandbox · **Reading order:** start here → `01` (ingestion + EV) → `02` (EDA + data dictionary)

Why we build on this dataset, in the **four-step modelling frame** from **W2D1**
([Blohm et al., 2019](https://doi.org/10.1523/ENEURO.0352-19.2019); tutorial `W2D1_ModelingPractice/W2D1_Tutorial1`,
pulled to `coursework/W2D1/`): *phenomenon → question → ingredients → hypothesis*, then the data.

*NMA references used as base:* the official fMRI project guide
([compneuro.neuromatch.io/projects/fMRI](https://compneuro.neuromatch.io/projects/fMRI/README.html)) and the loader
notebooks under `official/course-content/nma_course_content/projects/fMRI/`.

## Step 1 · Phenomenon & question

**Phenomenon.** When the brain moves from low working-memory load (**0-back**) to high load (**2-back**), the
functional coupling between cortical networks **reconfigures** — frontoparietal, cingulo-opercular and default
networks change how they communicate.

**Question (falsifiable).**
> Does functional-connectivity reconfiguration from 0-back to 2-back **predict** individual working-memory
> performance on **held-out** subjects?

The Project TA's north star is *prediction on unseen subjects*: connectivity and graph metrics are **features**, not
the goal, and the hypothesis stays yes/no testable.

## Step 2 · State of the art

Working memory engages the frontoparietal control and cingulo-opercular ("salience", in this atlas) networks, with
the default network typically anticorrelated under load. Our proof-of-concept (Goutham) already saw a salience →
FPN/DMN shift with load. *(Full literature review is a W2D1 task; this is the minimum framing for the ingredients below.)*

## Step 3 · Basic ingredients

Each data requirement is a clause of the hypothesis made measurable — Blohm Step 3. Derived from the
hypothesis sentence, before looking at any dataset.

In [1]:
from pathlib import Path
import os, sys
import numpy as np, pandas as pd

cwd = Path.cwd().resolve()
JAIME = cwd if (cwd / "datasets.py").exists() else cwd / "sandbox" / "jaime"
ROOT = JAIME.parents[1]                        # the-gammas repo root
sys.path.insert(0, str(JAIME))
import datasets as ds, preprocessing as pp, evaluation as ev
DATA = Path(os.environ.get("GAMMAS_DATA_DIR", ROOT / "data"))
A, B = ds.spec_a(DATA), ds.spec_b(DATA)
print(f"repo: {ROOT.name}  |  data: {DATA.name}/  |  datasets: N_PARCELS={ds.N_PARCELS}, TR={ds.TR}s")

repo: the-gammas  |  data: data/  |  datasets: N_PARCELS=360, TR=0.72s


In [2]:
# Requirements DERIVED from the hypothesis — not from any dataset yet.
requirements = pd.DataFrame([
    ("Working-memory task fMRI",    "clause '0-back vs 2-back'  ->  needs the N-back WM task",          "hard"),
    ("Parcellated ROI time series", "clause 'connectivity between regions'  ->  ROI time courses",      "hard"),
    ("A behavioural WM score",      "clause 'predict WM performance'  ->  the prediction TARGET",       "hard"),
    ("Many subjects",               "clause 'on held-out subjects'  ->  a sample across people",        "hard"),
    ("Resting-state (optional)",    "only if we keep abstract Objective 2 (intrinsic organization)",    "soft"),
], columns=["ingredient", "why (traces to a clause of the hypothesis)", "type"])
requirements

,ingredient,why (traces to a clause of the hypothesis),type
0,Working-memory task fMRI,clause '0-back vs 2-back' -> needs the N-bac...,hard
1,Parcellated ROI time series,clause 'connectivity between regions' -> ROI...,hard
2,A behavioural WM score,clause 'predict WM performance' -> the predi...,hard
3,Many subjects,clause 'on held-out subjects' -> a sample ac...,hard
4,Resting-state (optional),only if we keep abstract Objective 2 (intrinsi...,soft


Four **hard** requirements (impossible without them) and one **soft**. Every candidate dataset is
measured against this checklist.

## Step 4 · Hypothesis

**In words:** if load-driven FC reconfiguration encodes working-memory ability, then features derived from each
subject's 0→2-back change should predict their 2-back accuracy on subjects the model never saw.

**Made concrete:** target `y = acc_2bk` (2-back accuracy, one scalar per subject); features `X` = the 0→2-back FC
reconfiguration; evaluation = cross-validated prediction on **held-out subjects** (split by subject, never by frame).
A good target has spread across subjects and no missing values — checked below.

## Dataset landscape

NMA ships a set of fMRI loaders (see the official guide:
[projects/fMRI](https://compneuro.neuromatch.io/projects/fMRI/README.html), mirrored locally under
`official/course-content/nma_course_content/projects/fMRI/`). Measured against the Step-3 checklist, **most are
vision / stimulus-encoding studies** (responses to images or videos) and fail the first hard requirement (a WM task).

| Loader | Verdict | Reason (vs the Step-3 checklist) |
|---|---|---|
| `algonauts_videos` | reject | brain responses to video clips — no WM task, no target |
| `bonner_navigational_affordances` | reject | scene images + navigation ratings — not WM |
| `cichy_fMRI_MEG` | reject | visual-cortex RDMs to images — not WM |
| `fslcourse` | reject | fluency / finger-tapping — not WM, few subjects |
| `hcp_retino` | reject | retinotopy — averaged over subjects, no WM |
| `kay_images` | reject | responses to natural images — encoding, not WM |
| `hcp_task` | dominated | HCP WM task + 360 ROIs, but **no behaviour** shipped |
| **`hcp_task_with_behaviour`** | **finalist A** | HCP WM task + per-subject `Stats.txt` behaviour — 100 subjects |
| **`hcp`** | **finalist B** | HCP rest + task + behaviour covariates — 339 subjects |

> `hcp_task` is dominated by `hcp_task_with_behaviour` (same data **plus** the target), so the real contest is A vs B.

## The two finalists

Both finalists are downloaded locally under `data/` (gitignored). The decision hinges on one measurable
question: **is the prediction target present and well-spread in each?** (Full step-by-step EDA and B's
resting-state FC in `02`.)

In [3]:
# Concrete measurement behind the A-vs-B decision: is the TARGET present and well-spread in each finalist?
A_subs = ds.load_subjects(A)
behA = pp.behaviour_table(A)                       # target parsed from per-subject Stats.txt

wm = pd.read_csv(B.behaviour)          # Finalist B: consolidated behaviour
wm["load"] = wm["ConditionName"].str.extract(r"(\dBK)")
behB_acc2 = wm.groupby(["Subject", "load"])["ACC"].mean().unstack()["2BK"].dropna()

target_check = pd.DataFrame({
    "A · hcp_task_with_behaviour (Stats.txt)": [len(A_subs), int(behA.acc_2bk.isna().sum()),
        round(behA.acc_2bk.mean(), 3), round(behA.acc_2bk.min(), 3), round(behA.acc_2bk.max(), 3)],
    "B · hcp 339 (wm.csv)": [len(behB_acc2), 0,
        round(behB_acc2.mean(), 3), round(behB_acc2.min(), 3), round(behB_acc2.max(), 3)],
}, index=["n subjects", "missing acc_2bk", "mean", "min", "max"])
target_check

,A · hcp_task_with_behaviour (Stats.txt),B · hcp 339 (wm.csv)
n subjects,100.000,336.000
missing acc_2bk,0.000,0.000
mean,0.839,0.849
min,0.538,0.406
max,0.988,1.000


Both targets are usable and comparable (mean ≈ 0.84, wide spread, no missing values). B's consolidated
`wm.csv` carries the same per-condition ACC/RT as A's per-subject `Stats.txt` — it is not a coarser target.

In [4]:
# Head-to-head trade-off surface (the numbers that inform Monday's A-vs-B call)
compare = pd.DataFrame({
    "A — hcp_task_with_behaviour": ["100", "task only (7 tasks incl. WM)", "no",
        "Stats.txt · per condition, per subject", "already ingested (tasks 1-2 done)"],
    "B — hcp (339)": ["339", "task + rest (4 rest runs)", "yes",
        "wm.csv · per condition, consolidated", "would re-ingest on new subject IDs"],
}, index=["subjects", "acquisitions", "resting-state", "WM behaviour", "ingestion status"])
compare

,A — hcp_task_with_behaviour,B — hcp (339)
subjects,100,339
acquisitions,task only (7 tasks incl. WM),task + rest (4 rest runs)
resting-state,no,yes
WM behaviour,"Stats.txt · per condition, per subject","wm.csv · per condition, consolidated"
ingestion status,already ingested (tasks 1-2 done),would re-ingest on new subject IDs


## Decision (team · Monday)

Three defensible paths, each costed against the evidence above:

| Option | Meaning | Pros | Cons |
|---|---|---|---|
| **A** — stay on Finalist A (100) | Objectives 1+3 (task reconfiguration → performance) | ingestion done; per-subject `Stats.txt`; clean & falsifiable | 100 subjects; no rest; drops Obj. 2 |
| **B** — switch to Finalist B (339) | all 3 objectives incl. real rest FC | 3.4× subjects; resting-state; comparable target | re-ingest on new IDs; more work before Fri Jul 17 |
| **C** — stay on A, reframe Obj. 2 | keep A, treat intrinsic org. as a caveat | no rework; honest scope | Obj. 2 unaddressed empirically |

**Current base:** Finalist A (tasks 1–2 already run on it). **Not final** — B is attractive (more subjects + rest at
comparable target quality) for the cost of a scripted re-ingest. Monday should settle this; `02` gives the numbers.

## Method (reusable)

1. **Derive the data requirements from the hypothesis** (Step 3), not the reverse.
2. **List the full landscape** before rejecting anything (read the official guide rather than scripting it).
3. **Rule out against the requirements** — most candidates fail one hard requirement immediately.
4. **Measure the finalists** on the property the decision depends on (here, the target's presence and spread).
5. **Cost every option** — a capability available elsewhere still carries a switching cost; state it explicitly.